In [1]:
!git clone https://github.com/Nusultan11/imdb-sentiment.git
%cd /content/imdb-sentiment
!pip install -e .

Cloning into 'imdb-sentiment'...
remote: Enumerating objects: 526, done.
remote: Counting objects: 100% (526/526), done.
remote: Compressing objects: 100% (299/299), done.
remote: Total 526 (delta 274), reused 417 (delta 165), pack-reused 0 (from 0)
Receiving objects: 100% (526/526), 514.60 KiB | 14.70 MiB/s, done.
Resolving deltas: 100% (274/274), done.
/content/imdb-sentiment
Obtaining file:///content/imdb-sentiment
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for imdb-sentiment (pyproject.toml) ... done
  Created wheel for imdb-sentiment: filename=imdb_sentiment-0.1.0-0.editable-py3-none-any.whl size=5195 sha256=4a455c83b9eb56af9d03eed7f33633e9e681bbeb74b77d8cfbd94328c9f2f415
  Stored in directory: /tmp/pip-ephem-wheel-cache-vqsznqu5/wheels/3e/08/09/1f8f2dc306b20c0f6d4e5a4ddb080c355c7ce988d70820356f
Suc

In [22]:
import os
import gc
import json
import random
from dataclasses import replace
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn

import sys
sys.path.append("/content/imdb-sentiment/src")

from imdb_sentiment.data.dataset import load_imdb_dataset
from imdb_sentiment.data.lstm import build_lstm_dataloader, build_lstm_vocabulary
from imdb_sentiment.models.lstm.model import build_lstm_model
from imdb_sentiment.settings import load_config, LSTMModelConfig

import shutil
from google.colab import files

In [3]:
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))
else:
    print("running on CPU")

torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [4]:
!python -m imdb_sentiment.cli train --config configs/experiments/lstm_bidirectional_masked_mean_v1.yaml

README.md: 7.81kB [00:00, 20.6MB/s]
plain_text/train-00000-of-00001.parquet: 100% 21.0M/21.0M [00:00<00:00, 33.6MB/s]
plain_text/test-00000-of-00001.parquet: 100% 20.5M/20.5M [00:00<00:00, 49.7MB/s]
plain_text/unsupervised-00000-of-00001.p(…): 100% 42.0M/42.0M [00:00<00:00, 51.8MB/s]
Generating train split: 100% 25000/25000 [00:00<00:00, 146625.28 examples/s]
Generating test split: 100% 25000/25000 [00:00<00:00, 147827.30 examples/s]
Generating unsupervised split: 100% 50000/50000 [00:00<00:00, 170934.59 examples/s]
Training finished.
Accuracy: 0.8824
Precision: 0.8749
Recall: 0.8931
F1: 0.8839
Model saved to: /content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_v1/model.pt
Validation metrics saved to: /content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_v1/val_metrics.json


In [7]:
history_path = "/content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_v1/training_history.json"

with open(history_path, "r", encoding="utf-8") as f:
    payload = json.load(f)

print("best_epoch:", payload["best_epoch"])

history_df = pd.DataFrame(payload["history"])
print(history_df.to_string(index=False))

best_epoch: 3
 epoch  train_loss  val_loss   val_f1
     1    0.487168  0.385923 0.859703
     2    0.274400  0.293012 0.882662
     3    0.164561  0.307743 0.883886
     4    0.083843  0.465633 0.877358
     5    0.039355  0.452770 0.878173


In [8]:
threshold_path = "/content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_v1/threshold_tuning.json"
val_metrics_path = "/content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_v1/val_metrics.json"

with open(threshold_path, "r", encoding="utf-8") as f:
    threshold_payload = json.load(f)

with open(val_metrics_path, "r", encoding="utf-8") as f:
    val_metrics_payload = json.load(f)

print("threshold_tuning.json:")
print(json.dumps(threshold_payload, indent=2, ensure_ascii=False))

print("\nval_metrics.json:")
print(json.dumps(val_metrics_payload, indent=2, ensure_ascii=False))

threshold_tuning.json:
{
  "decision_threshold": 0.57,
  "selection_strategy": "validation_best_f1"
}

val_metrics.json:
{
  "loss": 0.3077430138067835,
  "accuracy": 0.8824,
  "precision": 0.8749022673964034,
  "recall": 0.8930566640063847,
  "f1": 0.8838862559241706
}


In [13]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

device: cuda
gpu: Tesla T4


In [17]:
BASE_CONFIG_PATH = "configs/experiments/lstm_bidirectional_masked_mean_v1.yaml"

base_config = load_config(BASE_CONFIG_PATH)
assert isinstance(base_config.model, LSTMModelConfig)

assert base_config.model.bidirectional is True
assert base_config.model.pooling == "masked_mean"

TUNING_EPOCHS = 5

print("Base experiment:", base_config.experiment.name)
print("Fixed architecture:")
print("  bidirectional =", base_config.model.bidirectional)
print("  pooling       =", base_config.model.pooling)
print("  epochs        =", TUNING_EPOCHS)

seed_everything(base_config.seed)

dataset = load_imdb_dataset()
train_val_split = dataset["train"].train_test_split(
    test_size=0.2,
    seed=base_config.seed,
)

train_split = train_val_split["train"]
val_split = train_val_split["test"]

x_train = list(train_split["text"])
y_train = [int(label) for label in train_split["label"]]

x_val = list(val_split["text"])
y_val = [int(label) for label in val_split["label"]]

print("train size:", len(x_train))
print("val size:", len(x_val))

vocabulary = build_lstm_vocabulary(
    texts=x_train,
    max_size=base_config.model.vocab_size,
)

print("vocab size:", len(vocabulary.token_to_id))

Base experiment: bidirectional_masked_mean_v1
Fixed architecture:
  bidirectional = True
  pooling       = masked_mean
  epochs        = 5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train size: 20000
val size: 5000
vocab size: 30000


In [14]:
def train_one_epoch(
    model: nn.Module,
    dataloader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    batch_count = 0

    for token_ids, labels in dataloader:
        token_ids = token_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(token_ids)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        batch_count += 1

    if batch_count == 0:
        raise ValueError("Training dataloader produced no batches.")

    return total_loss / batch_count


def collect_validation_outputs(
    model: nn.Module,
    dataloader,
    loss_fn: nn.Module,
    device: torch.device,
) -> tuple[float, list[int], list[float]]:
    model.eval()
    total_loss = 0.0
    batch_count = 0
    all_labels: list[int] = []
    all_probabilities: list[float] = []

    with torch.no_grad():
        for token_ids, labels in dataloader:
            token_ids = token_ids.to(device)
            labels = labels.to(device)

            logits = model(token_ids)
            loss = loss_fn(logits, labels)

            total_loss += float(loss.item())
            batch_count += 1

            probs = torch.sigmoid(logits).detach().cpu().tolist()
            all_probabilities.extend([float(p) for p in probs])
            all_labels.extend(labels.to(dtype=torch.int64).cpu().tolist())

    if batch_count == 0:
        raise ValueError("Validation dataloader produced no batches.")

    return total_loss / batch_count, all_labels, all_probabilities


def select_best_threshold(
    labels: list[int],
    probabilities: list[float],
) -> tuple[float, dict[str, float]]:
    if len(labels) != len(probabilities):
        raise ValueError("labels and probabilities must have the same length")
    if not labels:
        raise ValueError("validation set is empty")

    best_threshold = 0.5
    best_metrics = {"f1": -1.0}

    for threshold in [value / 100 for value in range(10, 91)]:
        predictions = [1 if prob >= threshold else 0 for prob in probabilities]

        precision, recall, f1, _ = precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            pos_label=1,
            zero_division=0,
        )
        accuracy = accuracy_score(labels, predictions)

        if f1 > best_metrics["f1"]:
            best_threshold = threshold
            best_metrics = {
                "accuracy": float(accuracy),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
            }

    return best_threshold, best_metrics

In [15]:
def objective(trial: optuna.Trial) -> float:
    trial_seed = base_config.seed + trial.number
    seed_everything(trial_seed)

    # search space
    embedding_dim = trial.suggest_categorical("embedding_dim", [64, 128, 256])
    hidden_dim = trial.suggest_categorical("hidden_dim", [64, 128, 256])
    dropout = trial.suggest_float("dropout", 0.10, 0.50, step=0.05)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    model_config = replace(
        base_config.model,
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        dropout=dropout,
        lr=lr,
        batch_size=batch_size,
        epochs=TUNING_EPOCHS,
    )

    train_loader = build_lstm_dataloader(
        texts=x_train,
        labels=y_train,
        vocabulary=vocabulary,
        max_length=model_config.max_length,
        batch_size=model_config.batch_size,
        shuffle=True,
        seed=trial_seed,
    )

    val_loader = build_lstm_dataloader(
        texts=x_val,
        labels=y_val,
        vocabulary=vocabulary,
        max_length=model_config.max_length,
        batch_size=model_config.batch_size,
        shuffle=False,
        seed=trial_seed,
    )

    model = build_lstm_model(model_config).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=model_config.lr)
    loss_fn = nn.BCEWithLogitsLoss()

    best_val_f1 = float("-inf")
    best_threshold = 0.5
    best_epoch = 0
    best_metrics = None

    try:
        for epoch_idx in range(model_config.epochs):
            train_loss = train_one_epoch(
                model=model,
                dataloader=train_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                device=device,
            )

            val_loss, val_labels, val_probabilities = collect_validation_outputs(
                model=model,
                dataloader=val_loader,
                loss_fn=loss_fn,
                device=device,
            )

            threshold, metrics = select_best_threshold(
                labels=val_labels,
                probabilities=val_probabilities,
            )

            current_val_f1 = metrics["f1"]

            if current_val_f1 > best_val_f1:
                best_val_f1 = current_val_f1
                best_threshold = threshold
                best_epoch = epoch_idx + 1
                best_metrics = {
                    "train_loss": float(train_loss),
                    "val_loss": float(val_loss),
                    **metrics,
                }

            trial.report(best_val_f1, step=epoch_idx)

            if trial.should_prune():
                raise optuna.TrialPruned()

        trial.set_user_attr("best_threshold", best_threshold)
        trial.set_user_attr("best_epoch", best_epoch)
        trial.set_user_attr("best_metrics", best_metrics)

        return best_val_f1

    finally:
        del model, optimizer, loss_fn, train_loader, val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [18]:
N_TRIALS = 15
STUDY_NAME = "imdb_lstm_bilstm_masked_mean_optuna"
STORAGE_URL = "sqlite:///optuna_lstm_bilstm_masked_mean.db"

sampler = optuna.samplers.TPESampler(seed=base_config.seed)
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=5,
    n_warmup_steps=2,
)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    load_if_exists=True,
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

print("Best trial number:", study.best_trial.number)
print("Best value (val_f1):", study.best_value)
print("Best params:", study.best_params)
print("Best user attrs:", study.best_trial.user_attrs)

[I 2026-03-31 09:09:05,715] A new study created in RDB with name: imdb_lstm_bilstm_masked_mean_optuna


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-03-31 09:11:52,846] Trial 0 finished with value: 0.8906930693069307 and parameters: {'embedding_dim': 128, 'hidden_dim': 64, 'dropout': 0.1, 'lr': 0.0029621516588303515, 'batch_size': 32}. Best is trial 0 with value: 0.8906930693069307.
[I 2026-03-31 09:15:11,323] Trial 1 finished with value: 0.8707831919510751 and parameters: {'embedding_dim': 64, 'hidden_dim': 256, 'dropout': 0.30000000000000004, 'lr': 0.0005418282319533242, 'batch_size': 32}. Best is trial 0 with value: 0.8906930693069307.
[I 2026-03-31 09:19:44,288] Trial 2 finished with value: 0.8443332691937656 and parameters: {'embedding_dim': 256, 'hidden_dim': 64, 'dropout': 0.35, 'lr': 0.00011992724522955161, 'batch_size': 16}. Best is trial 0 with value: 0.8906930693069307.
[I 2026-03-31 09:23:09,870] Trial 3 finished with value: 0.8266097750193949 and parameters: {'embedding_dim': 128, 'hidden_dim': 256, 'dropout': 0.25, 'lr': 0.000161190447276092, 'batch_size': 64}. Best is trial 0 with value: 0.8906930693069307.
[

In [19]:
CONFIG_PATH = Path("/content/imdb-sentiment/configs/experiments/lstm_bidirectional_masked_mean_optuna_v1.yaml")

config_text = """experiment:
  family: lstm
  name: bidirectional_masked_mean_optuna_v1

seed: 42

paths:
  model_output: artifacts/experiments/lstm/bidirectional_masked_mean_optuna_v1/model.pt
  val_metrics_output: artifacts/experiments/lstm/bidirectional_masked_mean_optuna_v1/val_metrics.json
  test_metrics_output: artifacts/experiments/lstm/bidirectional_masked_mean_optuna_v1/test_metrics.json

model:
  type: lstm
  vocab_size: 30000
  max_length: 512
  embedding_dim: 128
  hidden_dim: 128
  bidirectional: true
  pooling: masked_mean
  batch_size: 16
  epochs: 5
  dropout: 0.5
  lr: 0.002074566765675252
"""

CONFIG_PATH.write_text(config_text, encoding="utf-8")
print("saved config:", CONFIG_PATH)

saved config: /content/imdb-sentiment/configs/experiments/lstm_bidirectional_masked_mean_optuna_v1.yaml


In [20]:
!python -m imdb_sentiment.cli train --config /content/imdb-sentiment/configs/experiments/lstm_bidirectional_masked_mean_optuna_v1.yaml

Training finished.
Accuracy: 0.8954
Precision: 0.8990
Recall: 0.8915
F1: 0.8952
Model saved to: /content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_optuna_v1/model.pt
Validation metrics saved to: /content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_optuna_v1/val_metrics.json


In [21]:
artifact_dir = Path("/content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_optuna_v1")
val_metrics_path = artifact_dir / "val_metrics.json"
threshold_path = artifact_dir / "threshold_tuning.json"
history_path = artifact_dir / "training_history.json"

print("\nval_metrics.json")
print(val_metrics_path.read_text(encoding="utf-8"))

print("\nthreshold_tuning.json")
print(threshold_path.read_text(encoding="utf-8"))

print("\ntraining_history.json")
print(history_path.read_text(encoding="utf-8"))


val_metrics.json
{
  "loss": 0.35422696388615205,
  "accuracy": 0.8954,
  "precision": 0.8989939637826961,
  "recall": 0.8914604948124502,
  "f1": 0.8952113804848728
}

threshold_tuning.json
{
  "decision_threshold": 0.86,
  "selection_strategy": "validation_best_f1"
}

training_history.json
{
  "best_epoch": 2,
  "history": [
    {
      "epoch": 1,
      "train_loss": 0.40766156749129295,
      "val_loss": 0.26842679238071837,
      "val_f1": 0.8940345368916798
    },
    {
      "epoch": 2,
      "train_loss": 0.17246226621493696,
      "val_loss": 0.35422696388615205,
      "val_f1": 0.8952113804848728
    },
    {
      "epoch": 3,
      "train_loss": 0.05712745693651959,
      "val_loss": 0.3649415215417838,
      "val_f1": 0.8935240365514502
    },
    {
      "epoch": 4,
      "train_loss": 0.01971349293527528,
      "val_loss": 0.5886250742191907,
      "val_f1": 0.8815494393476044
    },
    {
      "epoch": 5,
      "train_loss": 0.010854043684381032,
      "val_loss": 0.51

In [23]:
artifact_dir = "/content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_optuna_v1"
archive_base = "/content/bidirectional_masked_mean_optuna_v1_artifacts"

shutil.make_archive(archive_base, "zip", artifact_dir)
files.download(f"{archive_base}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>